In [4]:
import numpy as np
import pymannkendall as mk
from scipy.stats import theilslopes
import os
import pandas as pd

def trend_analysis(df, col="VALOR", period=None, alpha=0.05):
    """
    Realiza análise de tendência temporal usando Mann-Kendall e Theil-Sen,
    registrando o período analisado, anos válidos e anos inválidos.

    Parâmetros
    ----------
    df : DataFrame
        DataFrame contendo pelo menos as colunas 'ano' e `col`.
    col : str, default="VALOR"
        Coluna com valores anuais da concentração.
    period : tuple (ano_ini, ano_fim) ou None
        Período da análise. Se None, usa o período completo do arquivo (ano.min()..ano.max()).
    alpha : float, default=0.05
        Nível de significância para o teste de Mann-Kendall.

    Retorno
    -------
    dict com:
        - start_year : int
        - end_year : int
        - n_years_valid : int
        - invalid_years : str (anos com NaN no período, separados por vírgula)
        - direction : str ("increasing", "decreasing", "no trend", "insufficient_data")
        - significant : bool
        - p_value : float
        - slope : float
        - median : float
        - percent_change : float (% de mudança relativo à mediana; mantém o sinal)
    """
    if "ano" not in df.columns:
        raise ValueError("A coluna 'ano' é obrigatória no DataFrame.")
    if col not in df.columns:
        raise ValueError(f"A coluna '{col}' não existe no DataFrame.")

    # define período
    if period is None:
        start_year = int(df["ano"].min())
        end_year   = int(df["ano"].max())
    else:
        start_year, end_year = map(int, period)

    # recorta período completo
    df_period = (
        df.loc[(df["ano"] >= start_year) & (df["ano"] <= end_year)]
          .sort_values("ano")
          .copy()
    )

    # anos inválidos (NaN) no período
    invalid_years_list = df_period.loc[df_period[col].isna(), "ano"].astype(int).tolist()
    invalid_years_str = ", ".join(map(str, invalid_years_list)) if invalid_years_list else ""

    # dados válidos para a análise
    df_valid = df_period.dropna(subset=[col]).copy()
    n_years_valid = int(df_valid.shape[0])

    # defaults
    direction = "insufficient_data"
    significant = False
    p_value = np.nan
    slope = np.nan
    median = np.nan
    percent_change = np.nan

    # só roda MK + Theil-Sen se houver dados suficientes (3 anos no mínimo)
    if n_years_valid >= 3:
        x = df_valid["ano"].values
        y = df_valid[col].values

        mk_result = mk.original_test(y, alpha=alpha)
        slope, intercept, _, _ = theilslopes(y, x)

        median = float(np.median(y))
        percent_change = float((slope / median) * 100) if median != 0 else np.nan

        if mk_result.trend == "increasing":
            direction = "increasing"
        elif mk_result.trend == "decreasing":
            direction = "decreasing"
        else:
            direction = "no trend"

        significant = bool(mk_result.p < alpha)
        p_value = float(mk_result.p)

    return {
        "start_year": start_year,
        "end_year": end_year,
        "n_valid_years": n_years_valid,
        "invalid_years": invalid_years_str,
        "direction": direction,
        "significance": significant,
        "p_value": p_value,
        "slope": slope,
        "median": median,
        "percent_change": percent_change,
    }


def trend_analysis_folder(folder, col="VALOR", period=None, file_ext=".csv"):
    """
    Aplica trend_analysis a todos os arquivos de uma pasta e gera um DataFrame.
    O nome do arquivo (sem extensão) vira o nome da estação na coluna 'station'.
    """
    results = []

    for file in os.listdir(folder):
        if file.endswith(file_ext):
            station_name = os.path.splitext(file)[0]
            filepath = os.path.join(folder, file)

            df = pd.read_csv(filepath)

            try:
                res = trend_analysis(df, col=col, period=period)
                res["station"] = station_name
                results.append(res)
            except Exception as e:
                print(f" Erro ao processar {file}: {e}")

    return pd.DataFrame(results)


folder_path = "/home/nobre/Notebooks/RQAR_2025_book/data/MQAr_averages/ano/O3"
df_results = trend_analysis_folder(folder_path, col="VALOR")
print(df_results)

   start_year  end_year  n_valid_years     invalid_years   direction  \
0        2005      2025             18  2005, 2024, 2025  decreasing   
1        2005      2025             18  2005, 2017, 2025  decreasing   
2        2005      2025             18  2005, 2007, 2025    no trend   
3        2005      2025             18  2005, 2022, 2025  decreasing   
4        2005      2025             18  2005, 2013, 2025  decreasing   

   significance   p_value     slope     median  percent_change      station  
0          True  0.000082 -0.849028  30.460373       -2.787321  SP0089RA001  
1          True  0.000059 -0.833528  29.940473       -2.783952  SP0092RA001  
2         False  0.058240 -0.993816  86.262764       -1.152080  SP0066RA001  
3          True  0.008015 -0.641166  31.722328       -2.021182  SP0087RA001  
4          True  0.000858 -0.688006  26.835217       -2.563818  SP0091RA001  


In [3]:
# -*- coding: utf-8 -*-

import os
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

import pymannkendall as mk
from scipy.stats import theilslopes

# -------------------------
# Configurações
# -------------------------
BASE_FOLDER = Path("/home/nobre/Notebooks/RQAR_2025_book/data/MQAr_averages/ano")
STATIONS_FILE = Path("/home/nobre/Notebooks/RQAR_2025_book/data/Monitoramento_QAr_BR.csv")
OUTPUT_DIR = Path("/home/nobre/Notebooks/RQAR_2025_book/_static/preprocessed_trends")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# lista de poluentes
POLLUTANTS = ["CO", "NO2", "SO2", "MP10", "MP25", "O3"]

# colunas na tabela de estações
ST_COL_ID = "ID_MMA_COMPLETO"
ST_COL_LAT = "LATITUDE"
ST_COL_LON = "LONGITUDE"

FILE_EXT = ".csv"
CRS_OUT = "EPSG:4326"

# -------------------------
def trend_analysis(df, col="VALOR", period=None, alpha=0.05):
    """
    Realiza análise de tendência temporal usando Mann-Kendall e Theil-Sen.
    Requer coluna 'ano' no DataFrame de entrada.
    Retorna dicionário com métricas de tendência.
    """
    if "ano" not in df.columns:
        raise ValueError("A coluna 'ano' é obrigatória no DataFrame.")
    if col not in df.columns:
        raise ValueError(f"A coluna '{col}' não existe no DataFrame.")

    # define período
    if period is None:
        start_year = int(df["ano"].min())
        end_year   = int(df["ano"].max())
    else:
        start_year, end_year = map(int, period)

    # recorta período completo
    df_period = (
        df.loc[(df["ano"] >= start_year) & (df["ano"] <= end_year)]
          .sort_values("ano")
          .copy()
    )

    # anos inválidos (NaN) no período
    invalid_years_list = df_period.loc[df_period[col].isna(), "ano"].astype(int).tolist()
    invalid_years_str = ", ".join(map(str, invalid_years_list)) if invalid_years_list else ""

    # dados válidos para a análise
    df_valid = df_period.dropna(subset=[col]).copy()
    n_years_valid = int(df_valid.shape[0])

    # defaults
    direction = "insufficient_data"
    significant = False
    p_value = np.nan
    slope = np.nan
    median = np.nan
    percent_change = np.nan

    # só roda MK + Theil-Sen se houver dados suficientes (3 anos no mínimo)
    if n_years_valid >= 3:
        x = df_valid["ano"].values
        y = df_valid[col].values

        mk_result = mk.original_test(y, alpha=alpha)
        slope, intercept, _, _ = theilslopes(y, x)

        median = float(np.median(y))
        percent_change = float((slope / median) * 100) if median != 0 else np.nan

        if mk_result.trend == "increasing":
            direction = "increasing"
        elif mk_result.trend == "decreasing":
            direction = "decreasing"
        else:
            direction = "no trend"

        significant = bool(mk_result.p < alpha)
        p_value = float(mk_result.p)

    return {
        "start_year": start_year,
        "end_year": end_year,
        "n_valid_years": n_years_valid,
        "invalid_years": invalid_years_str,
        "direction": direction,
        "significance": significant,
        "p_value": p_value,
        "slope": slope,
        "median": median,
        "percent_change": percent_change,
    }


def trend_analysis_folder(folder, col="VALOR", period=None, file_ext=".csv"):
    """
    Aplica trend_analysis a todos os arquivos de uma pasta e gera um DataFrame.
    O nome do arquivo (sem extensão) vira o nome da estação na coluna 'station'.
    """
    results = []
    folder = Path(folder)
    if not folder.exists():
        return pd.DataFrame(results)

    for file in sorted(folder.iterdir()):
        if file.is_file() and file.suffix.lower() == file_ext:
            station_name = file.stem
            filepath = str(file)

            try:
                df = pd.read_csv(filepath)
            except Exception as e:
                print(f"Falha ao ler {file.name}: {e}")
                continue

            try:
                res = trend_analysis(df, col=col, period=period)
                res["station"] = station_name
                res["source_file"] = file.name
                results.append(res)
            except Exception as e:
                print(f"Erro ao processar {file.name}: {e}")

    return pd.DataFrame(results)

# -------------------------
# Funções auxiliares
# -------------------------
def find_pollutant_folder(base_folder: Path, pol: str):
    """
    Retorna o Path da pasta do poluente se existir, caso contrário None.
    """
    if not base_folder.exists():
        return None
    for p in sorted(base_folder.iterdir()):
        if p.is_dir() and pol.lower() == p.name.lower():
            return p
        # permitir nomes com sufixos/extras
        if p.is_dir() and pol.lower() in p.name.lower():
            return p
    return None

def read_stations_meta(stations_file: Path):
    """
    Lê a tabela de estações e devolve DataFrame (com colunas ID_MMA_COMPLETO, LATITUDE, LONGITUDE quando disponíveis).
    """
    try:
        df = pd.read_csv(stations_file)
    except Exception as e:
        warnings.warn(f"Não foi possível ler {stations_file}: {e}")
        return pd.DataFrame(columns=[ST_COL_ID, ST_COL_LAT, ST_COL_LON])
    # garantir colunas esperadas
    for c in [ST_COL_ID, ST_COL_LAT, ST_COL_LON]:
        if c not in df.columns:
            df[c] = pd.NA
    # remover duplicatas mantendo a primeira aparição
    df = df.drop_duplicates(subset=[ST_COL_ID])
    return df[[ST_COL_ID, ST_COL_LAT, ST_COL_LON]]

def merge_with_station_coords(trend_df: pd.DataFrame, stations_meta: pd.DataFrame):
    """
    Faz merge simples left entre trend_df['station'] e stations_meta[ID_MMA_COMPLETO]
    Preenche apenas lat/lon que existam na tabela de estações.
    """
    if trend_df.empty:
        return trend_df
    if stations_meta.empty:
        # sem metadados, retorna trend_df com colunas lat/lon vazias
        trend_df[ST_COL_LAT] = pd.NA
        trend_df[ST_COL_LON] = pd.NA
        return trend_df

    merged = trend_df.merge(stations_meta, how="left", left_on="station", right_on=ST_COL_ID)
    # Se as colunas vierem com nomes diferentes, garantir lat/lon estão presentes
    if ST_COL_LAT not in merged.columns:
        merged[ST_COL_LAT] = pd.NA
    if ST_COL_LON not in merged.columns:
        merged[ST_COL_LON] = pd.NA
    return merged

def save_results_and_geojson(pol: str, merged_df: pd.DataFrame, out_dir: Path):
    """
    Salva CSV com resultados e GeoJSON com pontos (apenas para linhas com lat/lon).
    Retorna resumo para manifest.
    """
    pol_safe = pol.lower().replace(".", "").replace(" ", "_").replace(",", "")
    csv_path = out_dir / f"{pol_safe}_trend.csv"
    geojson_path = out_dir / f"{pol_safe}_stations.geojson"

    # salvar CSV 
    try:
        merged_df.to_csv(csv_path, index=False, encoding="utf-8")
    except Exception as e:
        print(f"Falha ao salvar CSV {csv_path}: {e}")

    # preparar GeoDataFrame apenas com estações que têm lat e lon preenchidos
    n_points = 0
    if ST_COL_LAT in merged_df.columns and ST_COL_LON in merged_df.columns:
        mask = merged_df[ST_COL_LAT].notna() & merged_df[ST_COL_LON].notna()
        pts = merged_df.loc[mask].copy()
        if not pts.empty:
            # converter para float
            try:
                pts[ST_COL_LAT] = pts[ST_COL_LAT].astype(float)
                pts[ST_COL_LON] = pts[ST_COL_LON].astype(float)
                pts["geometry"] = [Point(xy) for xy in zip(pts[ST_COL_LON], pts[ST_COL_LAT])]
                gdf = gpd.GeoDataFrame(pts, geometry="geometry", crs=CRS_OUT)
                gdf.to_file(geojson_path, driver="GeoJSON")
                n_points = len(gdf)
            except Exception as e:
                print(f"Falha ao gerar GeoJSON para {pol}: {e}")
                geojson_path.write_text(json.dumps({"type":"FeatureCollection","features":[]}, ensure_ascii=False))
                n_points = 0
        else:
            # pasta existe mas sem pontos com coords; criar geojson vazio
            geojson_path.write_text(json.dumps({"type":"FeatureCollection","features":[]}, ensure_ascii=False))
            n_points = 0
    else:
        # sem colunas lat/lon; criar geojson vazio
        geojson_path.write_text(json.dumps({"type":"FeatureCollection","features":[]}, ensure_ascii=False))
        n_points = 0

    return {"pol": pol.lower(), "csv": csv_path.name, "geojson": geojson_path.name, "n_points": int(n_points)}

# -------------------------
# Runner principal
# -------------------------
def process_all_pollutants(base_folder: Path = BASE_FOLDER, stations_file: Path = STATIONS_FILE,
                           pollutants=POLLUTANTS, out_dir: Path = OUTPUT_DIR, file_ext=FILE_EXT,
                           col="VALOR", period=None):
    manifest = {"generated": []}
    print("Base folder:", base_folder)

    # ler tabela de estações apenas uma vez
    stations_meta = read_stations_meta(stations_file)

    for pol in pollutants:
        pol_folder = find_pollutant_folder(base_folder, pol)
        if pol_folder is None:
            print(f"Pasta do poluente '{pol}' não encontrada em {base_folder}; pulando.")
            manifest["generated"].append({"pol": pol.lower(), "csv": None, "geojson": None, "n_points": 0})
            continue

        # confirmar se há arquivos CSV na pasta
        csv_files = [f for f in sorted(pol_folder.iterdir()) if f.is_file() and f.suffix.lower() == file_ext]
        if not csv_files:
            print(f"Pasta encontrada para '{pol}' ({pol_folder}), mas sem arquivos {file_ext}; pulando.")
            manifest["generated"].append({"pol": pol.lower(), "csv": None, "geojson": None, "n_points": 0})
            continue

        print(f"Processando poluente '{pol}' em: {pol_folder} (arquivos: {len(csv_files)})")

        # executar trend_analysis_folder
        trend_df = trend_analysis_folder(pol_folder, col=col, period=period, file_ext=file_ext)

        if trend_df.empty:
            print(f"Nenhum resultado válido para {pol} após análise; pulando geração de ficheiros.")
            manifest["generated"].append({"pol": pol.lower(), "csv": None, "geojson": None, "n_points": 0})
            continue

        # merge com metadados de estações para preencher lat/lon
        merged = merge_with_station_coords(trend_df, stations_meta)

        # salvar CSV e GeoJSON
        summary = save_results_and_geojson(pol, merged, out_dir)
        manifest["generated"].append(summary)
        print(f"{pol}: CSV -> {summary['csv']}, GeoJSON -> {summary['geojson']} (pontos: {summary['n_points']})")

    # salvar manifest
    manifest_path = out_dir / "manifest_trends.json"
    try:
        with open(manifest_path, "w", encoding="utf-8") as fh:
            json.dump(manifest, fh, ensure_ascii=False, indent=2)
        print("Manifest salvo em:", manifest_path)
    except Exception as e:
        print("Falha ao salvar manifest:", e)

    return manifest

# -------------------------
# Execução
# -------------------------
if __name__ == "__main__":
    manifest = process_all_pollutants()
    print("Concluído. Manifest:", manifest)



Base folder: /home/nobre/Notebooks/RQAR_2025_book/data/MQAr_averages/ano
⚠️ Pasta do poluente 'CO' não encontrada em /home/nobre/Notebooks/RQAR_2025_book/data/MQAr_averages/ano; pulando.
⚠️ Pasta do poluente 'NO2' não encontrada em /home/nobre/Notebooks/RQAR_2025_book/data/MQAr_averages/ano; pulando.
⚠️ Pasta do poluente 'SO2' não encontrada em /home/nobre/Notebooks/RQAR_2025_book/data/MQAr_averages/ano; pulando.
⚠️ Pasta do poluente 'MP10' não encontrada em /home/nobre/Notebooks/RQAR_2025_book/data/MQAr_averages/ano; pulando.
⚠️ Pasta do poluente 'MP25' não encontrada em /home/nobre/Notebooks/RQAR_2025_book/data/MQAr_averages/ano; pulando.
🔍 Processando poluente 'O3' em: /home/nobre/Notebooks/RQAR_2025_book/data/MQAr_averages/ano/O3 (arquivos: 5)
✅ O3: CSV -> o3_trend.csv, GeoJSON -> o3_stations.geojson (pontos: 5)
Manifest salvo em: /home/nobre/Notebooks/RQAR_2025_book/_static/preprocessed_trends/manifest_trends.json
Concluído. Manifest: {'generated': [{'pol': 'co', 'csv': None, 'geo